In [ ]:
#Spark connection
import os
import socket
from pyspark.sql import SparkSession
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = "/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar"
MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        getOrCreate()

In [ ]:
employees = [
    (1, "Scott", "Tiger", 1000.0, 
      "united states", "+1 123 456 7890", "123 45 6789"
    ),
     (2, "Henry", "Ford", 1250.0, 
      "India", "+91 234 567 8901", "456 78 9123"
     ),
     (3, "Nick", "Junior", 750.0, 
      "united KINGDOM", "+44 111 111 1111", "222 33 4444"
     ),
     (4, "Bill", "Gomes", 1500.0, 
      "AUSTRALIA", "+61 987 654 3210", "789 12 6118"
     )
]

In [ ]:
df_employees = spark. \
    createDataFrame(employees,
                    schema="""employee_id INT, first_name STRING, 
                    last_name STRING, salary FLOAT, nationality STRING,
                    phone_number STRING, ssn STRING"""
                   )

In [ ]:
df_employees.printSchema()

In [ ]:
df_employees.show(truncate=False)

In [ ]:
#Список всех функций. Обязательно посмотрите. Запомните где и как узнать весь список функций, получить описание и help
#https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html

from pyspark.sql.functions import date_format, col, lit, concat, concat_ws

In [ ]:
#Не забываем про help(имя_функции)
help(date_format)

In [ ]:
#Пример с groupBy

df_employees. \
    groupBy("nationality"). \
    count(). \
    show()

In [ ]:
#orderBy

df_employees. \
    orderBy("employee_id"). \
    show()

In [ ]:
from pyspark.sql.functions import upper

df_employees. \
    select(upper("first_name"), upper("last_name")). \
    show()

In [ ]:
df_employees. \
    select(upper(col("first_name")), upper(col("last_name"))). \
    show()

In [ ]:
#Обратите внимание на имя первой колонки
df_employees. \
    groupBy(upper(col("nationality"))). \
    count(). \
    show()

#Пример с использованием alias, чтобы убрать имя функции из имени колонки
df_employees. \
    groupBy(upper(col("nationality")).alias("Nationality")). \
    count(). \
    show()

#Добавляем orderBy
df_employees. \
    groupBy(upper(col("nationality")).alias("Nationality")). \
    count(). \
    orderBy(col("Nationality").asc()). \
    show()

In [ ]:
df_employees. \
    orderBy(col("employee_id").desc()). \
    show()

In [ ]:
df_employees. \
    orderBy(upper(df_employees['first_name']).alias('first_name')). \
    show()

In [ ]:
# Также мы можем обращаться к колонке таким образом df_name.col_name

df_employees. \
    orderBy(upper(df_employees.first_name).alias('first_name')). \
    show()

In [ ]:
from pyspark.sql.functions import concat, lit, col

df_employees. \
    select(col("*"), concat(col("first_name"), lit(", "), col("last_name")).alias("full_name")). \
    show()

In [ ]:
spark.stop()